In [59]:
import pandas as pd

df = pd.read_csv("hf://datasets/ailsntua/Chordonomicon/chordonomicon_v2.csv")
print(f"Loaded {len(df)} songs")

/tmp/ipykernel_4138/630165290.py:3: DtypeWarning: Columns (0: release_date, 1: genres, 2: rock_genre, 3: artist_id, 4: main_genre, 5: spotify_song_id, 6: spotify_artist_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("hf://datasets/ailsntua/Chordonomicon/chordonomicon_v2.csv")


Loaded 679807 songs


In [60]:
df.head()

,id,chords,release_date,genres,decade,rock_genre,artist_id,main_genre,spotify_song_id,spotify_artist_id
0,1,<intro_1> C <verse_1> F C E7 Amin C F C G7 C F...,NaN,'classic country pop',NaN,NaN,artist_1,pop,NaN,4AIEGdwDzPELXYgM5JaEY5
1,2,<intro_1> E D A/Cs E D A/Cs <verse_1> E D A/Cs...,2003-01-01,'alternative metal' 'alternative rock' 'nu met...,2000.0,pop rock,artist_2,metal,2ffJZ2r8HxI5DHcmf3BO6c,694QW15WkebjcrWgQHzRYF
2,3,<intro_1> Csmin <verse_1> A Csmin A Csmin A Cs...,2003-01-01,'alternative metal' 'canadian rock' 'funk meta...,2000.0,canadian rock,artist_3,metal,5KiY8SZEnvCPyIEkFGRR3y,0niJkG4tKkne3zwr7I8n9n
3,4,<intro_1> D Dmaj7 D Dmaj7 <verse_1> Emin A D G...,2022-09-23,NaN,2020.0,NaN,artist_4,NaN,01TtAcUqyLCRBZq4ZZiQWS,17BfKBemmMGO5ZAK25wraW
4,5,<intro_1> C <verse_1> G C G C <chorus_1> F Dmi...,2023-02-10,'modern country pop',2020.0,NaN,artist_5,pop,3zUecdrWC3IqrNSjhnoF3G,4GGfAshSkqoxpZdoaHm7ky


In [61]:
import ast

def load_mapping(path):
    df_map = pd.read_csv(path)
    df_map['Degrees'] = df_map['Degrees'].apply(ast.literal_eval)
    mapping = dict(zip(df_map['Chords'], df_map['Degrees']))
    return mapping

mapping = load_mapping("chords_mapping.csv")
print(f"Loaded mapping with {len(mapping)} chords")

Loaded mapping with 2793 chords


In [62]:
def normalize_chord(ch):
    ch = ch.strip().replace(" ", "")
    if not ch:
        return None
    chord = ch.replace("min", "m").replace("mi", "m")
    if "/" in chord:
        root, bass = chord.split("/", 1)
        bass = bass.replace("s", "#")
        return f"{root}/{bass}" if root else None
    replacements = [
        ("Cs", "C#"), ("Ds", "D#"), ("Es", "E#"),
        ("Fs", "F#"), ("Gs", "G#"), ("As", "A#"), ("Bs", "B#"),
    ]
    for old, new in replacements:
        chord = chord.replace(old, new)
    return chord if chord else None

print("normalize_chord('Csmin')  ->", normalize_chord("Csmin"))
print("normalize_chord('A/Cs')  ->", normalize_chord("A/Cs"))
print("normalize_chord('Emin')  ->", normalize_chord("Emin"))

normalize_chord('Csmin')  -> C#m
normalize_chord('A/Cs')  -> A/C#
normalize_chord('Emin')  -> Em


In [63]:
def smart_lookup(ch, mapping):
    if ch in mapping:
        return mapping[ch]
    variants = [
        ch.replace("m", "min"),
        ch.replace("#", "s"),
        ch.replace("m", "min").replace("#", "s"),
    ]
    for v in variants:
        if v in mapping:
            return mapping[v]
    return None

print("smart_lookup('Am', mapping)  ->", smart_lookup("Am", mapping))
print("smart_lookup('C', mapping)   ->", smart_lookup("C", mapping))

smart_lookup('Am', mapping)  -> [1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0]
smart_lookup('C', mapping)   -> [1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0]


In [64]:
import re

ROOT_PITCH = {'C': 0, 'C#': 1, 'Db': 1, 'D': 2, 'D#': 3, 'Eb': 3, 'E': 4,
              'Fb': 4, 'E#': 5, 'F': 5, 'F#': 6, 'Gb': 6, 'G': 7,
              'G#': 8, 'Ab': 8, 'A': 9, 'A#': 10, 'Bb': 10, 'B': 11, 'Cb': 11}

CHORD_INTERVALS = {
    '': [0, 4, 7], 'maj': [0, 4, 7], 'm': [0, 3, 7], 'min': [0, 3, 7],
    'dim': [0, 3, 6], 'aug': [0, 4, 8], 'sus2': [0, 2, 7], 'sus4': [0, 5, 7],
    '7': [0, 4, 7, 10], 'maj7': [0, 4, 7, 11], 'm7': [0, 3, 7, 10],
    'min7': [0, 3, 7, 10], 'dim7': [0, 3, 6, 9], 'aug7': [0, 4, 8, 10],
    '9': [0, 4, 7, 10, 2], 'maj9': [0, 4, 7, 11, 2], 'm9': [0, 3, 7, 10, 2],
    'add9': [0, 4, 7, 2], '6': [0, 4, 7, 9], 'm6': [0, 3, 7, 9], 'sus': [0, 5, 7],
}

def fallback_vector(chord_str):
    if '/' in chord_str:
        chord_str = chord_str.split('/')[0]
    match = re.match(r'^([A-G][#b]?)(.*)$', chord_str)
    if not match:
        return None
    root_str, quality = match.groups()
    root_str = root_str.replace("s", "#")
    if root_str not in ROOT_PITCH:
        return None
    root = ROOT_PITCH[root_str]
    quality_clean = quality.replace("min", "m")
    intervals = None
    for q, ints in CHORD_INTERVALS.items():
        if q and quality_clean == q:
            intervals = ints
            break
        if not q and quality_clean == '':
            intervals = ints
            break
    if intervals is None:
        for q, ints in CHORD_INTERVALS.items():
            if quality_clean.startswith(q):
                intervals = ints
                break
    if intervals is None:
        return None
    vector = [0] * 12
    for interval in intervals:
        vector[(root + interval) % 12] = 1
    return vector

print("fallback_vector('C')   ->", fallback_vector("C"))
print("fallback_vector('Am')  ->", fallback_vector("Am"))
print("fallback_vector('G7')  ->", fallback_vector("G7"))

fallback_vector('C')   -> [1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0]
fallback_vector('Am')  -> [1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0]
fallback_vector('G7')  -> [0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1]


In [65]:
def validate_vectors(vectors):
    for v in vectors:
        if len(v) != 12:
            return False
        if any(x not in [0, 1] for x in v):
            return False
    return True

def process_dataframe(df, mapping):
    all_vectors = []
    all_unknowns = []
    lengths = []
    for _, row in df.iterrows():
        prog = row['chords']
        tokens = prog.split()
        chords = []
        for t in tokens:
            if t.startswith("<"):
                continue
            norm = normalize_chord(t)
            if norm:
                chords.append(norm)
        vectors, unknown = chords_to_vectors(chords, mapping)
        all_vectors.append(vectors)
        all_unknowns.extend(unknown)
        lengths.append(len(vectors))
    df['vectors'] = all_vectors
    df['n_chords'] = lengths
    return df, list(set(all_unknowns))

def chords_to_vectors(chords, mapping):
    vectors = []
    unknown = []
    for ch in chords:
        vec = smart_lookup(ch, mapping) or fallback_vector(ch)
        if vec is not None and len(vec) == 12:
            vectors.append(vec)
        else:
            unknown.append(ch)
    return vectors, unknown

df_clean, unknowns = process_dataframe(df, mapping)

print(f"Songs processed: {len(df_clean)}")
print(f"Unknown chords: {len(unknowns)}")
if unknowns:
    print(f"Examples: {unknowns[:15]}")

df_clean = df_clean[df_clean['n_chords'] > 3]
df_clean = df_clean[df_clean['vectors'].apply(len) > 0]
df_clean = df_clean[df_clean['vectors'].apply(validate_vectors)]

print(f"After filtering (n_chords > 3, valid vectors): {len(df_clean)} songs")

Songs processed: 679807
Unknown chords: 1
Examples: ['sC']
After filtering (n_chords > 3, valid vectors): 679483 songs


In [66]:
def parse_sections(prog_str):
    tokens = prog_str.split()
    sections = {}
    current_section = None
    section_pattern = re.compile(r'^<([a-zA-Z]+)_\d+>$')
    for token in tokens:
        match = section_pattern.match(token)
        if match:
            current_section = match.group(1)
            if current_section not in sections:
                sections[current_section] = []
        elif current_section is not None and not token.startswith('<'):
            sections[current_section].append(token)
    return sections

def minimal_period_vectors(vectors):
    n = len(vectors)
    for k in range(1, n + 1):
        if n % k != 0:
            continue
        pattern = vectors[:k]
        if pattern * (n // k) == vectors:
            return k
    return n

def process_sections(df, mapping):
    rows = []
    section_counter = {}
    all_unknowns = set()
    for idx, row in df.iterrows():
        song_id = row['id']
        prog_str = row['chords']
        sections = parse_sections(prog_str)
        for section_name, raw_chords in sections.items():
            normalized = [normalize_chord(ch) for ch in raw_chords if normalize_chord(ch)]
            if len(normalized) < 3:
                continue
            vectors = []
            valid_chords = []
            for ch in normalized:
                vec = smart_lookup(ch, mapping) or fallback_vector(ch)
                if vec is not None and len(vec) == 12:
                    vectors.append(vec)
                    valid_chords.append(ch)
                else:
                    all_unknowns.add(ch)
            if len(valid_chords) < 3:
                continue
            k = minimal_period_vectors(vectors)
            base_vectors = vectors[:k]
            base_chords = valid_chords[:k]
            repeats = len(vectors) // k
            if section_name not in section_counter:
                section_counter[section_name] = 0
            section_counter[section_name] += 1
            section_id = f"{song_id}_{section_name}_{section_counter[section_name]}"
            rows.append({
                'song_id': song_id,
                'section': section_name,
                'section_id': section_id,
                'chords': base_chords,
                'vectors': base_vectors,
                'n_chords': len(base_chords),
                'repeats': repeats,
                'original_len': len(valid_chords)
            })
    return pd.DataFrame(rows), all_unknowns

print("Processing sections...")
df_sections, unknown_chords = process_sections(df_clean, mapping)

print(f"\nTotal section rows: {len(df_sections)}")
print(f"Unique section types: {sorted(df_sections['section'].unique())}")
print(f"Unknown chords (fallback used): {len(unknown_chords)}")
if unknown_chords:
    print(f"Examples: {list(unknown_chords)[:10]}")

print(f"\nSection distribution:")
print(df_sections['section'].value_counts())

print(f"\nExample row:")
if len(df_sections) > 0:
    ex = df_sections.iloc[0]
    print(f"  song_id: {ex['song_id']} | section: {ex['section']} | n_chords: {ex['n_chords']} | repeats: {ex['repeats']}")
    print(f"  chords: {ex['chords'][:5]}...")
    print(f"  vectors[0]: {ex['vectors'][0]}")

Processing sections...

Total section rows: 1339089
Unique section types: ['bridge', 'chorus', 'instrumental', 'interlude', 'intro', 'outro', 'solo', 'verse']
Unknown chords (fallback used): 1
Examples: ['sC']

Section distribution:
section
verse           368732
chorus          332059
intro           214117
bridge          147988
outro           146021
instrumental     51004
interlude        47884
solo             31284
Name: count, dtype: int64

Example row:
  song_id: 1 | section: verse | n_chords: 17 | repeats: 2
  chords: ['F', 'C', 'E7', 'Am', 'C']...
  vectors[0]: [1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0]


In [67]:
df_sections.head()

,song_id,section,section_id,chords,vectors,n_chords,repeats,original_len
0,1,verse,1_verse_1,"[F, C, E7, Am, C, F, C, G7, C, F, C, E7, Am, C...","[[1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0], [1, 0, ...",17,2,34
1,1,chorus,1_chorus_1,"[F, C, F, C, G, C, F, C, E7, Am, C, F, G7, C, ...","[[1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0], [1, 0, ...",31,1,31
2,2,intro,2_intro_1,"[E, D, A/C#]","[[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1], [0, 0, ...",3,2,6
3,2,verse,2_verse_2,"[E, D, A/C#, E, D, A/C#, E, D, A/C#, E, D, A, C]","[[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1], [0, 0, ...",13,2,26
4,2,chorus,2_chorus_2,"[E, G, D, A, E, G, D, A, E, G, D, A, C, D, E, ...","[[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1], [0, 0, ...",45,1,45


In [ ]:
df_sections[['chords', 'vectors']].to_parquet(
    "dataset.parquet",
    engine="fastparquet",
    compression="gzip",
    index=False
)